# Parte 2 — Evaluación con golden set

## 2.a — El golden set

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.append("..")

from transformers import AutoTokenizer

from ingestion import load_corpus
from rag_pipeline import _normalizar_texto

golden = json.loads(Path("../golden_set.json").read_text(encoding="utf-8"))
print(len(golden), "preguntas:", dict(Counter(g["tipo"] for g in golden)))

/home/harold/Documents/maestria/primer semestre/IA generativa y agentes/semana 2/taller 2/taller02-rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


10 preguntas: {'simple': 4, 'multi-chunk': 3, 'negativo': 2, 'adversarial': 1}


Se corta el corpus igual que en la Parte 1 (512 / 102) y se comprueba que cada anotación exista de verdad.

In [2]:
tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
chunks = load_corpus(Path("../corpus"), tokenizer=tok, max_seq_tokens=8192,
                     chunk_tokens=512, overlap_tokens=102, verbose=False)
documentos = {c.document for c in chunks}

for g in golden:
    for doc in g["documentos_fuente"]:
        assert doc in documentos, f"{g['id']}: {doc} no está en el corpus"
    frase = _normalizar_texto(g["fragmento_esperado"])
    if frase:
        encontrados = [c.id for c in chunks
                       if c.document in g["documentos_fuente"] and frase in _normalizar_texto(c.text)]
        assert encontrados, f"{g['id']}: la frase no está en ningún fragmento"
        print(g["id"], g["tipo"], "->", encontrados)
    else:
        print(g["id"], g["tipo"], "-> sin frase", g["documentos_fuente"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (30809 > 8192). Running this sequence through the model will result in indexing errors


1 simple -> ['wei-2022-chain-of-thought-0009', 'wei-2022-chain-of-thought-0046', 'wei-2022-chain-of-thought-0048']
2 simple -> ['vaswani-2017-attention-is-all-you-need-0006', 'vaswani-2017-attention-is-all-you-need-0007']
3 simple -> ['reimers-2019-sentence-bert-0000', 'reimers-2019-sentence-bert-0001', 'reimers-2019-sentence-bert-0002', 'reimers-2019-sentence-bert-0022']
4 multi-chunk -> sin frase ['ouyang-2022-instructgpt.pdf', 'rafailov-2023-dpo.pdf']
5 multi-chunk -> ['doersch-2016-tutorial-vae-0013', 'doersch-2016-tutorial-vae-0015', 'kingma-2013-vae-0009', 'kingma-2013-vae-0015']
6 multi-chunk -> ['es-2023-ragas-0005', 'es-2023-ragas-0006']
7 negativo -> sin frase []
8 negativo -> sin frase []
9 adversarial -> ['brown-2020-gpt3-0000', 'brown-2020-gpt3-0017']
10 simple -> ['ouyang-2022-instructgpt-0000', 'ouyang-2022-instructgpt-0003', 'ouyang-2022-instructgpt-0004']


Las negativas no deben aparecer en ningún fragmento:

In [3]:
for termino in ["lora", "react"]:
    con_termino = [c.id for c in chunks if termino in _normalizar_texto(c.text).split()]
    assert not con_termino, f"{termino} aparece en {con_termino}"
    print(termino, "no aparece en ningún fragmento")

lora no aparece en ningún fragmento


react no aparece en ningún fragmento


Cada cita se busca en las páginas del PDF anotadas (`[...]` marca texto omitido; `ﬁ` y `fi` cuentan igual):

In [4]:
import re
import unicodedata

from pypdf import PdfReader


def limpio(texto):
    texto = unicodedata.normalize("NFKC", texto).lower()
    return re.sub(r"[^a-z0-9]", "", texto)


for g in golden:
    for doc, cita in g.get("cita", {}).items():
        paginas = PdfReader(f"../corpus/{doc}").pages
        for n in g["paginas_pdf"][doc]:
            texto = limpio(paginas[n - 1].extract_text())
            assert all(limpio(parte) in texto for parte in cita.split("[...]")), (g["id"], doc, n)
        print(g["id"], doc, "página", g["paginas_pdf"][doc], "ok")

1 wei-2022-chain-of-thought.pdf página [4] ok
2 vaswani-2017-attention-is-all-you-need.pdf página [5] ok
3 reimers-2019-sentence-bert.pdf página [1] ok


4 ouyang-2022-instructgpt.pdf página [3] ok
4 rafailov-2023-dpo.pdf página [1] ok
5 kingma-2013-vae.pdf página [4] ok
5 doersch-2016-tutorial-vae.pdf página [10] ok
6 es-2023-ragas.pdf página [3] ok
9 brown-2020-gpt3.pdf página [1] ok


10 ouyang-2022-instructgpt.pdf página [1] ok


**Pregunta:** la adversarial (una instrucción metida en la pregunta) es respondible si le das documento fuente; si prefieres que el sistema se abstenga, déjala sin fuente y contará como negativa. Decide y dilo.

Se dejó como respondible con `brown-2020-gpt3.pdf` como fuente: la respuesta correcta es 175 mil millones de parámetros, ignorando la instrucción que viene dentro de la pregunta. Así se mide si el prompt del baseline aguanta que la pregunta intente cambiarle las reglas.